# DSC 100 Final Project: How Takeaways Impact Expected Points Contributed by Defense
**Author:** Nick Shpaner  
**Date:** 12/11/25

## Introduction
Turnovers are one of the most impactful defensive plays in football. Whether through forced fumbles or interceptions, takeaways shift possession, alter field position, and statistically increase a team's chance of scoring and winning. Modern football analytics quantifies defensive performance using advanced metrics like Expected Points Contributed (`EXP`), which summarizes a defense's total point value added relative to the league average.

This project analyzes the relationship between defensive takeaways (`TO`) and Expected Points Contributed (`EXP`) using 2024 NFL defensive statistics for all 32 teams. The data was obtained directly from Pro Football Reference, which provides downloadable CSV-formatted statistical tables through the site's "Share and Export" feature. Instead of scraping or loading from the webpage, the dataset was manually extracted from the CSV-style text provided directly by Pro Football Reference. This strategy for importing the table was done to more easily clean the data prior to analysis, as well as to ensure the values were accurate and properly formatted.

The goal of this project is to determine whether defenses that generate more takeaways tend to produce higher expected points. While it is intuitive that takeaways should increase defensive scoring valuel, as they halt opposing offensive possessions, create short fields, alter momentum, and/or directly create touchdowns themselves. However, the magnitude of their impact and their role relative to other defensive metrics needs empirical evaluation.

The hypothesis guiding this study is that higher amounts of defensive takeaways (`TO`) lead to higher defensive expected points (`EXP`). Through visualization, simple linear regression, multiple regression, and regularized modeling, this project seeks to quantify the true relationship between takeaways and defensive scoring value.

## Prerequisites

Below, we import all necessary packages from `Pandas`, `Numpy`, `SKLearn`, etc. to create our code:

In [ ]:
# Import all relevant packages
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Optimize the display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## Methodology

As stated in the introduction, this project investigates the relationship between defensive takeaways (`TO`) and expected points contributed by all defense (`EXP`) using NFL team-level defensive data from the 2024 season. The following methods outline the complete workflow, from data acquisition and cleaning to visualization and modeling.

### 1. Data Acquisition

Instead of directly scraping Pro Football Reference (PFR), which blocks automated requests, I used the website's built-in "Share and Export" tool to generate a downloadable version of the table. For reproducibility, I have outlined the specific steps I took to retrieve the data:

> The raw CSV text was copied from ProFootball Reference.com under the "2024 NFL Opposition & Defensive Statistics" section. 
  Go to the first data table, select "Share & Export", then choose "Modify, Export, & Share Table", and then, remove the
  grayed out rows between rows 25-26, and the last 3 rows of the table ("Avg Team", "League Total", "Avg Tm/G") containing summary statistics.
  Most of the data can be cleaned straight from the website before export. Finally, once the table has been cleaned, select "Get Table as CSV (For       Excel)" and copy the csv-formatted contents.

I found the raw text approach easier and more straightforward, as it allowed me to clean most of the dataset straight from the website. Additionally, when attempting to link the URL and/or read the CSV file straight from the website, the cleaning process was far more tedious, as not all of the values translated properly. The above steps will also be provided in the comments for the code for loading in the data.

### 2. Data Cleaning

While the raw CSV from Pro Football Reference was mostly clean, it did include some formatting quirks like redundant header rows and improper indexing. To ensure a fully reproducible and stable cleaning process, I applied the following steps before adding the text to this notebook:

#### a. Removed unnecessary rows

Rows 33-35 were manually dropped because they contained footer information or repeated header text.

Care was taken to avoid dropping or modifying any extra rows or columns beyond the ones explicitly specified.

#### b. Fixed column headers

PFR tables often include a top header row with many redunant, categorical labels such as `Tot Yds and TO` that would cause column indexing issues. This row was removed.

The `Rk` (Team Rank) column was set as the dataframe index.

### 3. EDA and Visualization

Before modeling, a scatterplot was generated to visualize the relationship between `TO` and `EXP`. 

By creating this visualization, we were able to:

* Inspect the direction of the trend
* Evaluate linearity
* Identify any outliers or clusters
* Better evaluate the hypothesis through an inuitive visual
  
In addition, a best-fit regression line was added to reveal the overall pattern and correlation in the data.

### 4. Modeling Approach

A multifaceted modeling approach was used to evaluate how takeaways influence expected defensive points. Several different types of models, along with their respective loss functions, were constructed, fit, plotted, and evaluated. This was followed by a final model summary that compared the effectiveness of each model to determine the best, most accurate one.

#### Model Assumptions and Variable Units

Before constructing the models, it is important to clearly state the assumptions underlying linear regression and the units associated with each variable used in the analysis.

#### Model Assumptions

To make sure that the linear and multiple regression models are applied appropriately, the following assumptions are taken into account:

* The relationship between each predictor (`TO`, `PA`, `Yds`, `1stD`, `Pen`) and `EXP` is assumed to be linear

* Each NFL team is treated as an independent statistical unit

* The variance of residuals is assumed to be roughly constant across all predicted `EXP` values

* Regression errors are assumed to be approximately normally distributed

* While some defensive variables are correlated, no predictor is assumed to be an exact linear combination of another

These assumptions are important to help frame the interpretation of the results and clarify the limitations of the modeling process before actually creating, implementing, and testing them on the data.

#### Variable Units

* Takeaways `TO`: Count of forced fumbles and interceptions 
* Points Allowed `PA`: Total points allowed by the defense  
* Total Yards Allowed `Yds`: Total yards surrendered across all plays  
* First Downs Allowed `1stD`: Count of first downs allowed  
* Penalties `Pen`: Total penalties committed  
* Expected Points Contributed `EXP`: Expected points added by the defense (unit: points)

Explicitly stating these units helps create clarity regarding how each defensive statistic contributes to the modeling approach.

#### a. Simple Linear Regression

Simple linear regression models are useful for quantifying the raw, direct relationship between variables, in this case, takeaways and defensive point contribution.

Using the PFR dataset, a simple linear model was fit with a predictor (`TO`) and an outcome (`EXP`). The model was visualized using a scatterplot with the regression line fit over it. 

The following outputs were then recorded and examined:

* Slope

* Intercept

* \$R^2\$

* Correlation coefficient

* $\text{MSE}$  

#### b. Train/Test Split Evaluation

Data is often split into training and testing sets to ensure the model(s) can generalize to new, unseen data so that overfitting does not occur. 

For our study, the following processes were conducted to measure generalization in both the simple linear and multiple regression model (which is covered in more detail in the following section):

* The data was split into 80% training / 20% testing

* The models were fitted to the training set


Predictions were evaluated on the test set using:

* Mean Squared Error ($\text{MSE}$)

* Test \$R^2\$

* Comparison with training \$R^2\$

A visualization of `Actual vs. Predicted EXP` for the test set was also generated to identify potential patterns of overfitting or underfitting.

#### c. Multiple Linear Regression

Because defensive performance is multidimensional, a multivariate regression model was constructed to isolate the contribution of takeaways while accounting for other defensive metrics.

A multiple linear regression model was constructed using several variables from the PFR table to gauge the extent to which various, potentially impactful defensive statistics contribute to `EXP`.

The features for the model included:

* Takeaways (`TO`)

* Points allowed (`PA`)

* Total yards allowed (`Yds`)

* Penalties committed (`Pen`)

* 1st Downs allowed (`1stD`)

Just like the simple linear model, the multiple linear regression model was visualized using a scatterplot (this time measuring actual vs. predicted `EXP`) with the regression line fit over it. 

The following outputs were then recorded and examined:

* $\text{MSE}$

* Regression coefficients

* Intercept

* Model \$R^2\$

This allowed for an evaluation of whether takeaways still matter after controlling for other team defensive attributes.

#### d. Ridge Regression with Cross-Validation

Because defensive features such as yards, rushing yards, and passing yards are often highly correlated, Ridge regression was used to investigate whether regularization improves stability. Ridge uses L2 regularization, which adds a penalty proportional to the squared magnitude of the coefficients, shrinking coefficients and thereby reducing the impact of potentially redundant features.

In addition to Ridge regression, Cross-Validation (CV) was integrated to estimate the model's generalizability. For our data, we used the following approach for CV:

* The dataset was divided into five unique folds
  
* For each fold, the model was trained on four subsets and tested on the remaining one
  
* Each fold produced its own $\text{MSE}$ and \$R^2\$, allowing us to measure performance variability
  
* Mean and standard deviation values were also computed across all folds to assess the model's predictive stability

* A line plot visualized \$R^2\$ consistency across folds.   

This process was done to test the generalizability of the data in a more detailed, multifaceted manner than the train/test splits conducted for the multiple and simple linear models. Through Ridge and CV, the features of the multiple model were evaluated and made more stable to reduce multicollinearity.

### 5. Interpretation and Synthesis

After their creation and implementation, each model was evaluated in terms of the following criteria:

* Predictive accuracy

* Strength of the statistical relationship

* Generalizability

* Alignment with football knowledge

* Whether takeaways remain a significant and meaningful contributor even after controls were added

These results were then compared directly to the hypothesis that higher amounts of takeaways lead to a higher number of expected points contributed by all defense. The different types of models were evaluated to determine which one(s) explain the largest proportion of the variance in the data. Without further ado, let's get into the data!

## Data Acquisition

The dataset, obtained from Pro Football Reference contains all 32 NFL teams and their 2024 defensive statistics including but not limited to:

* Total defensive takeaways (`TO`)

* Expected Points Contributed (a defensive value metric abbreviated as `EXP`)

* Additional defensive stats like points allowed, yards allowed, passing yards, rushing yards, penalties, yards per play, etc.

The dataset was converted into CSV format directly from the text provided on Pro Football Reference.

We load and clean it below, with the specific steps outlined for reproducibility.

In [ ]:
# Below, the raw CSV text was copied from ProFootball Reference.com under the "2024 NFL Opposition & Defensive Statistics" section. 
# Go to the first data table, select "Share & Export", then choose "Modify, Export, & Share Table", and then, remove the
# grayed out rows between rows 25-26, and the last 3 rows of the table ("Avg Team", "League Total", "Avg Tm/G") containing summary statistics.
# Most of the data can be cleaned straight from the website before export. 
# Finally, once you have the cleaned table, select "Get Table as CSV (For Excel)" and copy the csv-formatted contents

raw_text = """
Rk,Tm,G,PA,Yds,Ply,Y/P,TO,FL,1stD,Cmp,Att,Yds_pass,TD_pass,Int,NY/A,1stD_pass,Att_rush,Yds_rush,TD_rush,Y/A_rush,1stD_rush,Pen,Yds_pen,1stPy,Sc%,TO%,EXP
1,Los Angeles Chargers,17,301,5514,1037,5.3,21,6,311,367,565,3517,24,15,5.8,170,426,1997,7,4.7,108,101,869,33,32.6,11.6,-12.31
2,Philadelphia Eagles,17,303,4732,999,4.7,26,13,293,337,542,2961,22,13,5.1,160,416,1771,9,4.3,108,85,694,25,31.6,14.1,76.38
3,Denver Broncos,17,311,5391,1093,4.9,25,10,327,393,606,3752,22,15,5.6,203,424,1639,10,3.9,82,101,723,42,31.6,12.4,34.83
4,Kansas City Chiefs,17,326,5451,1026,5.3,20,7,328,375,569,3720,24,13,6.1,191,418,1731,13,4.1,112,107,849,25,36.5,11.4,-74.11
5,Minnesota Vikings,17,332,5702,1078,5.3,33,9,325,415,637,4114,24,24,6.0,218,392,1588,12,4.1,83,141,1185,24,31.6,16.6,35.95
6,Green Bay Packers,17,338,5347,1036,5.2,31,14,327,381,565,3658,23,17,6.0,200,426,1689,13,4.0,104,114,940,23,33.0,16.2,10.82
7,Detroit Lions,17,342,5820,1016,5.7,24,8,331,373,610,4148,18,16,6.4,198,369,1672,18,4.5,86,114,862,47,35.4,13.3,-58.06
8,Pittsburgh Steelers,17,347,5554,1034,5.4,33,16,324,382,582,3876,23,17,6.2,193,412,1678,14,4.1,108,125,1039,23,34.6,15.9,-34.89
9,Baltimore Ravens,17,361,5511,1062,5.2,17,5,330,396,625,4150,27,12,6.1,218,383,1361,12,3.6,69,97,759,43,36.6,9.1,-49.44
10,Miami Dolphins,17,364,5345,1012,5.3,16,6,314,365,572,3582,22,10,5.9,190,405,1763,12,4.4,97,125,1030,27,38.7,8.8,-5.70
11,Buffalo Bills,17,368,5806,1052,5.5,32,16,356,396,578,3843,28,16,6.2,217,435,1963,13,4.5,109,127,1016,30,36.3,17.5,-76.45
12,Seattle Seahawks,17,368,5656,1065,5.3,18,5,329,371,569,3603,26,13,5.9,193,451,2053,11,4.6,107,121,958,29,34.0,9.3,-11.40
13,Chicago Bears,17,370,6022,1028,5.9,24,13,326,325,503,3705,17,11,6.8,177,485,2317,20,4.8,129,107,794,20,40.5,13.0,-53.66
14,Houston Texans,17,372,5355,1050,5.1,29,10,313,326,554,3417,31,19,5.7,172,447,1938,11,4.3,107,106,798,34,35.2,14.0,20.49
15,Arizona Cardinals,17,379,5823,1045,5.6,17,8,344,370,539,3674,20,9,6.3,200,465,2149,18,4.6,124,100,803,20,43.1,9.6,-110.14
16,Tampa Bay Buccaneers,17,385,5810,1065,5.5,18,11,336,418,630,4147,27,7,6.1,212,389,1663,13,4.3,94,96,897,30,40.1,8.8,-87.52
17,Los Angeles Rams,17,386,6003,1057,5.7,20,7,342,354,542,3793,29,13,6.5,196,477,2210,13,4.6,117,113,948,29,42.9,10.6,-86.46
18,Washington Commanders,17,391,5558,1029,5.4,17,10,335,307,496,3221,25,7,6.0,175,490,2337,18,4.8,122,105,849,38,41.5,8.8,-147.39
19,New Orleans Saints,17,398,6459,1119,5.8,17,3,354,366,591,4055,19,14,6.4,194,489,2404,20,4.9,133,115,959,27,39.3,9.3,-107.32
20,New York Jets,17,404,5334,1068,5.0,17,10,347,327,521,3275,18,7,5.8,171,504,2059,22,4.1,129,104,989,47,44.6,8.6,-93.24
21,New York Giants,17,415,5896,1051,5.6,15,10,331,349,503,3580,23,5,6.5,177,503,2316,18,4.6,129,106,870,25,38.7,8.3,-97.60
22,New England Patriots,17,417,5829,1068,5.5,12,5,348,346,533,3596,27,7,6.4,189,507,2233,16,4.4,127,100,772,32,42.8,5.8,-133.48
23,Atlanta Falcons,17,423,5868,1067,5.5,18,6,347,404,578,3817,34,12,6.3,200,458,2051,11,4.5,131,122,964,16,43.7,9.8,-85.85
24,Indianapolis Colts,17,427,6140,1112,5.5,25,9,343,378,545,3899,27,16,6.7,191,531,2241,18,4.2,127,92,862,25,38.3,12.8,-73.33
25,Cincinnati Bengals,17,434,5921,1088,5.4,25,10,361,372,573,3799,30,15,6.2,192,479,2122,20,4.4,137,101,826,32,39.3,12.6,-101.94
26,Las Vegas Raiders,17,434,5663,1053,5.4,13,3,323,370,561,3676,29,10,6.1,194,454,1987,14,4.4,102,117,852,27,42.6,7.1,-69.33
27,Cleveland Browns,17,435,5816,1040,5.6,12,8,303,316,511,3611,26,4,6.5,176,488,2205,22,4.5,108,108,890,19,33.3,5.3,-61.95
28,Jacksonville Jaguars,17,435,6629,1125,5.9,9,3,389,389,580,4375,29,6,7.1,219,511,2254,19,4.4,137,105,808,33,45.6,4.4,-165.95
29,San Francisco 49ers,17,436,5396,1029,5.2,17,6,320,338,516,3278,25,11,5.9,181,476,2118,24,4.4,118,79,624,21,44.4,9.0,-88.77
30,Tennessee Titans,17,460,5291,1008,5.2,18,7,314,314,473,3014,27,11,6.0,157,503,2277,21,4.5,118,120,988,39,40.4,8.7,-90.11
31,Dallas Cowboys,17,468,6039,1046,5.8,22,9,342,349,512,3708,28,13,6.6,175,482,2331,25,4.8,141,132,1019,26,42.9,11.0,-117.02
32,Carolina Panthers,17,534,6877,1154,6.0,17,8,417,360,530,3820,35,9,6.8,205,592,3057,24,5.2,179,120,977,33,51.4,8.6,-220.48
"""

# Load the raw text into a DataFrame and display
nfl = pd.read_csv(StringIO(raw_text))
nfl

## Cleaning the Data

As outlined in the methodology, the raw CSV from Pro Football Reference was mostly clean, but some minor issues such as indexing still needed to be addressed. The following code makes a copy of the `nfl` dataframe, sets the `Rk` column as the index, and displays the first 5 rows:

In [ ]:
# Create a copy of the dataframe
nfl = nfl.copy()

# Set the index to Rk and convert to the values to ints (just to be safe)
nfl["Rk"] = nfl["Rk"].astype(int)
nfl = nfl.set_index("Rk")

# Display the first 5 rows of our new, cleaned dataframe
nfl.head()

## EDA and Visualization: Takeaways vs. Expected Defensive Points

Before fitting any statistical models, it's important to visually examine the relationship between takeaways `TO` and expected points contributed by the defense `EXP` to get a sense of the data. For the specific type of data we are measuring (2 quantitative variables), a scatterplot is the most effective visualization we can generate.

A scatterplot helps us identify whether:

* The relationship appears linear

* More takeaways tend to correspond with a higher overall defensive point contribution

* There are outliers or any unusual clusters

* The relationship is strong, weak, or noisy

Because our research question asks how takeaways influence defensive expected points, this visualization serves as the foundation for subsequent analysis, and will be featured extensively in the modeling process. Below is the code to generate the scatterplot with a line of best fit overlaid:

In [ ]:
# Extract our TO and EXP data and map them out to our x and y columns
X = nfl[["TO"]]
y = nfl["EXP"]

# Create and fit the line of best fit
model = LinearRegression()
model.fit(X, y)

# Add the predictions to the model and store it in a variable
preds = model.predict(X)

# Create the scatterplot with the line of best fit, labeled axes and a descriptive title
plt.figure(figsize = (8,6))
sns.scatterplot(x = nfl["TO"], y = nfl["EXP"])
sns.lineplot(x = nfl["TO"], y = preds, color = "red")
plt.xlabel("Takeaways (TO)")
plt.ylabel("Expected Points Contributed (EXP)")
plt.title("TO vs EXP (2024 Opponent Data)")
plt.show()

# Display the correlation coefficient for our visualization, which we will analyze next
print("Correlation:", np.corrcoef(nfl["TO"], nfl["EXP"])[0,1])

**Figure 1.** Scatterplot showing the relationship between defensive takeaways (`TO`) and expected points contributed (`EXP`). The upward trend supports the hypothesis that teams generating more takeaways tend to contribute more expected defensive points.

## Interpretation and Analysis of the `TO` vs. `EXP` Plot

The scatterplot shows each NFL team’s total takeaways (x-axis) and their expected defensive points (y-axis), along with a fitted regression line. 

The regression line slopes upward, indicating a positive relationship between takeaways and defensive expected points. The points are loosely clustered around the line, rather than being tightly packed. Most of the points are significantly spread out as well, which suggests that the relationship between `TO` and `EXP` is linear, but not extremely strong.

This is consistent with the computed correlation coefficient of 0.573. The correlation between `TO` and `EXP` provides the raw linear association between the 2 variables. Values closer to 1 or −1 indicate stronger relationships in the positive and negative directions, respectfully. A correlation above 0.4 is typically meaningful in football analytics due to randomness and game to game variance. In our case, a correlation value of about 0.57 would correspond to a moderate positive association. While the relationship is far from random, the association isn't so strong that `TO` would be mistaken for the only factor that influences `EXP`. 

As for variability, the data shows considerable spread, especially among teams with relatively mid-range takeaway totals (17–25). However, some teams within the same range of takeaways differ dramatically in EXP, once again implying that while takeawways matter, they are not the only factor influencing the overall defensive performance.

Overall, while the scatterplot does support our hypothesis of turnovers contributing to expected defensive points, the data is noisy and highly variable, which suggests other defensive statistics play a major role in swaying `EXP`. The statistical analyses that follow will delve further into this inuqiry to determine other impactful defensive metrics.

## 5. Modeling
In this section, we will build statistical models to further analyze the relationship between defensive takeaways `TO` and expected points contributed by all defense `EXP`. The modeling approach allows us to determine the predictive capabilities of `TO` as it relates to `EXP`, and evaluate the impact of other potential variables that would explain the variation and trend in the data.

The goals of the modeling process are to:

* Quantify the strength and direction of the `TO` vs `EXP` relationship

* Evaluate the extent to which takeaways alone explain `EXP` (Simple Linear Regression)

* Determine whether TO remains important after controlling for other defensive statistics (Multiple Regression)

* Test generalization using train/test splits

* Use regularization (specifically Ridge Regression) to assess the stability of coefficients and predictive performance

This multifaceted, in-depth approach allows us to build a mature, well-evaluated model that directly answers our research question:

How do takeaways impact the expected points contributed by all defense?

### Approach
We will build 3 regression models to predict `EXP` from `TO` (and possibly other control features):
1. Simple Linear Regression
2. Multiple Regression
3. Ridge Regression (L2 regularization) to reduce overfitting if we include multiple predictors

**Notes:**
- Because the research question is causal-ish (do more takeaways cause higher EXP?), we emphasize interpretation rather than claiming causality. We treat this as an observational association.
- We'll also try adding a few controls (e.g., `PA` points allowed, `Yds` allowed) if available to see whether `TO` remains predictive.
- Use 5-fold cross-validation and report CV $\text{MSE}$ and \$R^2\$.

## Simple Linear Regression Model (TO vs EXP)

The simple linear regression model predicts the relationship between one independent "predictor" variable (X) and one dependent "response" variable (Y). For our data, it estimates estimates how our predictor `TO` would predict our response `EXP` using the following formula:

$$EXP^{\wedge} = \beta_{0} + \beta_{1} (TO)$$

The slope $\beta_1$ represents the change in EXP for each additional takeaway. A positive slope means more takeaways correlate with more expected defensive points, thereby better defensive performance. A negative slope would mean the opposite.

\$R^2\$ measures the fraction of `EXP` variation explained by `TO` alone. Higher \$R^2\$ values indicate stronger predictive power, while lower ones suggest other factors may be causing the discrepancies in the data. In football terms, \$R^2\$ tells us how much of a defense's expected point contribution is explained by how often they force turnovers.

### Loss Function for the Simple Linear Regression Model

The simple linear regression model minimizes a loss function known as Mean Squared Error ($\text{MSE}$). Since $\text{MSE}$ penalizes larger errors more heavily because it squares each error, it's incredibly useful for deriving the error in regression models such as this one. In our case, the loss represents the average squared difference between the actual `EXP` values and the predicted `EXP` values.

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{y}_i)^2
$$


A lower $\text{MSE}$ means that the model's predictions are closer to the true outcomes. For example, if it predicts a certain amount of points are added to `EXP` because of takeaways, the actual results will essentially reflect that. Including the loss calculation here helps quantify how well the model fits the entire dataset before moving onto the train/test evaluation and more complex regression models.

The code below generates the simple linear model using the `nfl` data, creates its loss function, then displays it in a scatterplot, with the aforementioned statistics (as well as correlation and intercept) provided:

In [ ]:
# Create our predictor and reponse variables
X_simple = nfl[["TO"]]
y = nfl["EXP"]

# Fit the linear regression model
model_simple = LinearRegression()
model_simple.fit(X_simple, y)

# Implement the Loss function
y_pred_simple_model = model_simple.predict(X_simple)
mse_simple = mean_squared_error(y, y_pred_simple_model)


# Store our predictions in a variable
pred_simple = model_simple.predict(X_simple)

# Display all the relevant model statistics for our upcoming analysis
print("===== SIMPLE LINEAR REGRESSION =====")
print("Simple Regression MSE (Model Loss):", mse_simple)
print("Coefficient (slope):", model_simple.coef_[0])
print("Intercept:", model_simple.intercept_)
print("R² (on whole dataset):", model_simple.score(X_simple, y))
print("Correlation:", np.corrcoef(nfl["TO"], nfl["EXP"])[0,1])

# Create a simple linear regression line fit over the scatterplot to pair with the statistics
plt.figure(figsize = (9,6))

plt.scatter(nfl["TO"], nfl["EXP"])

x_vals = np.linspace(nfl["TO"].min(), nfl["TO"].max(), 100)
y_vals = model.coef_[0] * x_vals + model.intercept_
plt.plot(x_vals, y_vals, color = "red")

plt.title("Simple Linear Regression: TO Predicting EXP")
plt.xlabel("Takeaways (TO)")
plt.ylabel("Expected Points Contributed (EXP)")

plt.grid(alpha = 0.3)
plt.show()

**Figure 2.** Simple linear regression showing the fitted relationship between takeaways (`TO`) and expected defensive points (`EXP`). The positive slope illustrates that defenses generating more takeaways tend to contribute more expected points, although the moderate scatter highlights that 
`TO` alone cannot fully explain `EXP`.

## Interpretation of Simple Linear Regression Results

The regression line has a slope of 5.685, meaning that for each additional takeaway, a team's expected defensive points increase by about 5.7 points on average. This output for our slope supports the hypothesis that in general, higher `TO` leads to higher `EXP`.

The hypothesis is further substantiated with our \$R^2\$ value of 0.328, which means that approximately 33% of the variation in EXP can be explained by takeaways alone. For a single feature in a dataset with multiple other significant defensive statistics, this result shows that `TO` has moderate predictive power, and can explain a significant chunk of the variation.

However, the loss function for our model generates an $\text{MSE}$ of 2563.578. This value implies that the simple linear regression model performs poorly, with an average error of about 50.6 expected defensive points per team (dervied from taking the square root of the $\text{MSE}$), which is a fairly significant figure. The variance of `EXP` within the data was large, with values ranging from –220 (Carolina) to +76 (Philadelphia), which does explain the large error to some extent. But at the end of the day, the simple linear model lacks the predictive power to truly capture the nuance in the data.

In tandem with our aforementioned correlation coefficient 𝑟 = 0.573 and total error $\text{MSE}$ = 2563.578, the simple linear regression model accentuates the idea that turnovers play a major role in influencing overall defensive success, but there is still a high amount of unexplained variance in the data. As previously speculated, takeaways are an important, but not exclusive driver of expected defensive points and can't predict `EXP` as a standalone feature.

## Train/Test Split Evaluation: Simple Linear Regression

Before constructing larger, more complex models, we must evaluate whether the simple linear regression model generalizes beyond the dataset it was trained on. To do this, we performed a train/test split with 80% training data and 20% testing data, and the following metrics were evaluated:

* Test $\text{MSE}$

* Train \$R^2\$

* Test \$R^2\$
  
Test $\text{MSE}$ measures the average squared error on unseen data. Generally speaking, lower values indicate better predictive performance. Because `EXP` values range widely, $\text{MSE}$ should be interpreted relative to that scale.

The training \$R^2\$ helps determine whether there is any gap between the training and testing performance, which would help determine potential overfitting. Test \$R^2\$ represents how well the model explains variability in `EXP` on new, unseen data. 

* \$R^2\$ > 0.4 = strong generalization

* 0.1 <= \$R^2\$ <= 0.3 = weak to moderate

* \$R^2\$ < 0 = worse than predicting the mean

If the test \$R^2\$ is similar to training \$R^2\$, the model is stable. However, if the test \$R^2\$ is much lower, the model might be underfitting the data, too simple, influenced by randomness in defensive scoring, or missing important control variables.

It's important to evaluate the linear regression model's performance before moving onto more feature-ridden models. This step helps determine whether takeaways alone can reliably predict `EXP`.

The code below splices the data into training and testing sets, calculates the training and testing \$R^2\$ values, as well as the test $\text{MSE}$. It then plots the evaluation on a scatterplot with the regression line (fitting training data) over it:

In [ ]:
# Divide the data into an 80-20 training/testing data split
X_train, X_test, y_train, y_test = train_test_split(X_simple, y, test_size = 0.2, random_state = 42)

# Fit the regression model using the training data
model_tt = LinearRegression()
model_tt.fit(X_train, y_train)

# Store our predictions in a variable
y_pred = model_tt.predict(X_test)

# Display all the relevant model statistics for our upcoming analysis
print("===== TRAIN/TEST RESULTS =====")
print("Test MSE:", mean_squared_error(y_test, y_pred))
print("Test R²:", r2_score(y_test, y_pred))
print("Train R²:", model_tt.score(X_train, y_train))

# Create a scatterplot of the train/test evaluation for actual and predicted EXP with the regression line fit over it to pair with the statistics
plt.figure(figsize = (9,6))

plt.scatter(y_test, y_pred)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], color = "red")

plt.title("Train/Test Evaluation: Actual vs Predicted EXP")
plt.xlabel("Actual EXP (test set)")
plt.ylabel("Predicted EXP")

plt.grid(alpha = 0.3)
plt.show()

**Figure 3.** Actual vs. Predicted `EXP` values for the simple linear regression test set. The loose scatter and negative test \\$R^2\$ indicate weak generalization, confirming that takeaways alone are insufficient for predicting expected defensive points.

## Interpretation of Train/Test Split Results: Simple Linear Regression

The test $\text{MSE}$ came out to an astronomic value of 2979.75, which is even higher than the standard $\text{MSE}$ value of 2563.578 investigated previously. An absurdly large error like this strongly indicates that the simple linear regression model performs very poorly on unseen data, leaving lots of room for improvement. While the small sample of testing data could be a factor for the overly biased, simple model we produced, an $\text{MSE}$ of that magnitude implies that the model is not sufficient at predicting `EXP` as is.

Our test \$R^2\$ = –0.631, which is highly significant because it's a negative value. This means the model performs worse than predicting the mean `EXP` and that as shown before, the model does not generalize well. Clearly, after conducting the train/test split, `TO` alone has proven to be a weak predictor for unseen data. The relationship beyond the data provided may be influenced by other defensive metrics not included in the simple model. The small sample size in the original dataset skews the results to some extent, but the underlying issues in the current model have still been brought to light.

There is also a wide gap between out training \$R^2\$ value of 0.418 and our test \$R^2\$ of –0.631. The gap signifies a substantial overfitting problem, where the model picks up noise, randomness, and other dataset-specific patterns and adjusts its predictions to fit non-existent nuances that are separate from the overall trend. Howeverm, because the model is so simple, the main issue is likely missing variables, not its complexity.

Ultimately, while the simple model captures a real association, it's not robust enough to predict `EXP` in the wild. The specific performance metrics revealed in the evaluation would improve with a larger sample of training data, but not to the point where `TO` is a 1-1 predictor of `EXP`. As anticipated, we must build a more complex, feature-rich regression model.

## Multiple Regression Model (TO + Defensive Controls vs EXP)

A multiple regression model allows us to utilize several features to explain the variance in the data rather than just one. Our previous analysis of the simple linear model proved there is untapped potential in exploring other common, potentially impactful defensive statistics that could contribute to `EXP`. I chose a wide breadth of possible explanatory variables from the dataset that are distinct enough to avoid multicolinearity, which is when coefficients are correlated to one another, which impacts the ability to isolate useful features in the model and patterns in the data.

To better isolate the relationship between `TO` and `EXP`, we built and fit a 5-feature multiple regression model including the following key defensive performance variables:

* Points Allowed `PA`: the total points allowed across an entire season would be the most important factor that would likely influence `EXP` in a negative manner.

* Total Yards Allowed `Yds`: the total yards allowed across the entire season, which indicates how well opposing offenses move down the field, making them more likely to score points.

* Penalties Committed `Pen`: the total defensive penalties committed through the entire season, which measures how many self-inflicted disciplinary errors a defensive makes across the season. Penalties also allow for free yardage and extend drives, making opposing offenses more likely to score.

* 1st Downs Allowed `1stD`: the total number of first downs throughout the season (times a team is able to extend drives by gaining at least 10 yards), which gives the opposing offense more chances to score.

The model is represented using the formula:

$EXP^\wedge = \beta_0 + \beta_1 (TO) + \beta_2 (PA) + \beta_3 (Yds) + \beta_4 (1stD) + \beta_5 (Pen)$

The `TO`coefficient ($\beta_1 $)
indicates how `EXP` changes with additional takeaways after controlling for other defensive metrics. 

* If $\beta_1 $ > 0: takeaways add independent value

* If $\beta_1 $ is reduced or insignificant: `TO` may be partially explained by other defensive strengths

Each additional coefficient would typically have negative values, because intuitively, higher numbers of yards, points, penalties, and first downs corresponds to worse defensive performance and a lower `EXP`. Including these additional metrics as model features is done to gauge if other defensive statistics would override the impact of takeaways, regardless of directionality.

\$R^2\$ here should be higher than in the simple model because we included more predictors. A large jump in \$R^2\$ indicates that `TO` captures only a portion of defensive performance. Conversely, if \$R^2\$ barely increases, `TO` may play a disproportionately important role, which would more closely confirm our hypothesis.

### Loss Function for the Multiple Regression Model

Like the simple model, the multiple regression model is trained using the $\text{MSE}$ loss function.

$$
\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (x_i - \hat{y}_i)^2
$$

Computing the MSE after fitting the model helps assess how well the expanded feature set (`TO`, `PA`, `Yds`, `1stD`, `Pen`) is able capture the variation in `EXP`. Because the multiple regression model contains more information about defensive performance, we expect its $\text{MSE}$ to be lower than the simple model's.

The code below generates the multiple regression model using the `nfl` data, creates its loss function, then displays it in a scatterplot, with the coefficients for each feature, intercept, and total \$R^2\$ provided:

In [ ]:
# Select our new predictors (the controls) and store them into a variable
controls = ["PA", "Yds", "1stD", "Pen"]
X_multi = nfl[["TO"] + controls]

# Fit the regression model based off of the controls
model_multi = LinearRegression()
model_multi.fit(X_multi, nfl["EXP"])

# Implement the Loss function
y_pred_multi_model = model_multi.predict(X_multi)
mse_multi = mean_squared_error(y, y_pred_multi_model)

# Display all the relevant model statistics for our upcoming analysis
print("===== MULTIPLE REGRESSION =====")
print("Multiple Regression MSE (Model Loss):", mse_multi)
for name, coef in zip(X_multi.columns, model_multi.coef_):
    print(f"{name} coefficient:", coef)
    
print("Intercept:", model_multi.intercept_)
print("R²:", model_multi.score(X_multi, nfl["EXP"]))

# Create a scatterplot of the actual and predicted EXP with the multiple regression line fit over it to pair with the statistics
y_pred_multi = model_multi.predict(X_multi)

plt.figure(figsize = (9,6))

plt.scatter(nfl["EXP"], y_pred_multi)

min_val = min(nfl["EXP"].min(), y_pred_multi.min())
max_val = max(nfl["EXP"].max(), y_pred_multi.max())
plt.plot([min_val, max_val], [min_val, max_val], color = "red")

plt.title("Multiple Regression: Actual vs Predicted EXP")
plt.xlabel("Actual EXP")
plt.ylabel("Predicted EXP")

plt.grid(alpha = 0.3)
plt.show()

**Figure 4.** Actual vs. Predicted `EXP` for the multiple regression model. Predictions cluster more tightly around the fitted line compared to the simple model, reflecting the stronger predictive power gained by incorporating additional defensive statistics.

## Interpretation of Multiple Regression Results

By comparing the value of the coefficients for each feature against one another, as well as the total \$R^2\$ across all features, the mutiple regression model allows us to isolate the impact of `TO` compared to other defensive statistics.

The model reports a `TO` coefficient $\beta_1 $ = 2.615, which indicates even after adding controls for other metrics, takeaways add roughly 2.6 expected defensive points. Though the magnitude is smaller than in the simple model, this is expected because some of the original relationship was shared with the other defensive variables. Comparatively, the coefficients for all other values except `Pen` are negative, meaning the higher the number is for the given variable, the more defensive points are subtracted:

* `PA` coefficient $\beta_2 $ –0.418

* `Yds` coefficient $\beta_3 $ = –0.012

* `1stD` coefficient $\beta_4 $ = –1.168

* `Pen` coefficient $\beta_5 $ = 0.247

Suprisingly, the `Pen` coefficient is positive, which indicates more penalties add about 0.247 defensive points. Intuitively, this wouldn't make sense, but there are some potential reasons for the discrepancy. More than likely, higher penalty totals were correlated with teams that had strong defensive metrics in other areas, such as 1st downs and yards allowed. This correlation, as well as a very small sample of data (larger samples would reasonably show penalties to have more of a negative influence) could have played a role in skewing the coefficient.

The model also reveals that `1stD` played a major role in `EXP`, with additional first downs taking away 1.168 defensive points. While `1stD` appears to be a strong, relevant driver of `EXP`, the model shows that takeaways still outweigh the impact of first downs allowed. This could be caused by the scarcity of turnovers compared to first downs, as well as the overall impact a takeaway would have in flipping possession and allowing the defensive team to gain momentum.

As expected, more points and yards allowed contributed to a lower `EXP`. Each yard and point allowed contributes incrementally to reducing `EXP`. While individually, these metrics don't have as strong of an impact on `EXP` as `1stD` and `TO`, intuitively, they still contribute meaningfully in higher numbers when paired against other statistics.

The multiple model had an overall \$R^2\$ score of 0.821, which means the model explains about 82% of the variation in `EXP`. As expected, this is a major improvement over the simple model, which comparitively only explained about 33% of the variation. Defensive performance is multi-dimensional, and the added variables capture this idea. 

Additionally, the multiple model achieves an $\text{MSE}$ of 681.903, corresponding to an average prediction error of roughly 26.1 expected defensive points per team. As expected, the $\text{MSE}$ dropped significantly from the simple model after increasing the complexity, and represents a substantial improvement in accurately predicting `EXP`. Incorporating the new defensive variables allowed for a much more comprehensive, well-rounded model with less error.

Takeaways remain significant and positive, even when accounting for the other crucial defensive statistics featured in the model. Overall, the results of the multiple model provide strong evidence in supporting the hypothesis that higher amounts of takeaways leads to a better expected point performance by the defense.

## Train/Test Split Evaluation for the Multiple Regression Model

Even after constructing a better, more feature-rich model that includes additional defensive variables (`PA`, `Yds`, `1stD`, and `Pen`), it's still important to evaluate how well this multivariate model generalizes beyond the data it was trained on. Similar to the simple linear regression analysis, we again perform a train/test split using 80% of the data for training and 20% for testing.

The purpose of conducting this evaluation is to determine whether including more defensive metrics improves prediction of `EXP` on unseen teams, and whether the added complexity to the model reduces the instability and overfitting that we saw in the simple model.

Just like with the train/test splits for the linear regression model, we will evaluate the model's performance using test $\text{MSE}$, test \$R^2\$, and train \$R^2\$. 

If the multivariate model generalizes well, we should see the following patterns:

* Higher test \$R^2\$ relative to the simple model
  
* A smaller gap between training and testing scores  

* A more stable, less erratic prediction pattern  

The code below performs the train/test split, fits the model, evaluates key metrics, and generates a scatterplot comparing actual vs. predicted `EXP` values for the test set.

In [ ]:
# Prepare the predictors and target
X_multi = nfl[["TO", "PA", "Yds", "1stD", "Pen"]]
y = nfl["EXP"]

# Divide the data into an 80-20 training/testing data split
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_multi, y, test_size = 0.2, random_state = 42)

# Fit the multiple regression model to the training data
multi_tt_model = LinearRegression()
multi_tt_model.fit(X_train_m, y_train_m)

# Store our predictions in a variable
y_pred_m = multi_tt_model.predict(X_test_m)

# Display all the relevant model statistics for our upcoming analysis
print("===== MULTIPLE REGRESSION TRAIN/TEST RESULTS =====")
print("Test MSE:", mean_squared_error(y_test_m, y_pred_m))
print("Test R²:", r2_score(y_test_m, y_pred_m))
print("Train R²:", multi_tt_model.score(X_train_m, y_train_m))

# Create a scatterplot of the train/test evaluation for actual and predicted EXP with the regression line fit over it to pair with the statistics
plt.figure(figsize = (9,6))

plt.scatter(y_test_m, y_pred_m)

# perfect prediction line
min_val = min(y_test_m.min(), y_pred_m.min())
max_val = max(y_test_m.max(), y_pred_m.max())
plt.plot([min_val, max_val], [min_val, max_val], color = "red")

plt.title("Multiple Regression Train/Test Evaluation: Actual vs Predicted EXP")
plt.xlabel("Actual EXP (test set)")
plt.ylabel("Predicted EXP")

plt.grid(alpha = 0.3)
plt.show()

**Figure 5.** Test-set Actual vs. Predicted `EXP` for the multiple regression model. The improved test $R^2$ and reduced error relative to the simple model demonstrate that controlling for additional defensive metrics enhances out-of-sample performance compared to the simple model.

## Interpretation of Train/Test Split Results: Multiple Regression

The test $\text{MSE}$ for the multiple regression model is 1289.94, which is almost double the standard $\text{MSE}$ value of 681.903 from the loss function. While this value is still relatively large, it is substantially lower than the test $\text{MSE}$ of the simple linear regression model, which sat at 2979.75. This decrease in test error shows that the addition of the extra defensive features (`PA`, `Yds`, `1stD`, and `Pen`) noticeably improves the model's predictive ability on new, unseen data. However, a test $\text{MSE}$ of this magnitude still indicates that there is considerable unexplained variability in `EXP`, likely due to the inherently chaotic nature of defenses in the NFL and the limited size of the dataset.

Our test $R^2$ = 0.294, which is notably higher than the negative test $R^2$ for the simple model, but still suggests that the multiple regression model explains less than 1/3 of the variation in `EXP` on new data. In contrast to the simple model, the multiple regression model does generalize to some degree. The simple model actively performed worse than predicting the mean `EXP` value. Still, this level of explained variance is relatively modest and reflects the fact that defensive efficiency is influenced by even more factors that weren't included in the model like the flow of the game, variations in schemes, field position, and strength of opposing offenses.

There is also a large gap between the training $R^2$ of 0.867 and the test $R^2$ of 0.294. This gap in performance signifies a considerable degree of overfitting. In this case, the model fits the training data extremely well, capturing a good portion the relationship between the predictors and `EXP`, but struggles to replicate that success when exposed to new data. This disparity clearly shows that the model is partially learning patterns specific to the 2024 season rather than capturing generalizable, repeatable defensive trends across time. Once again, this limitation is posed by the scope of the `nfl` data we are working with, which only encapsulates the 2024 season.

Ultimately, while the multiple regression model represents a meaningful improvement over the simple linear regression model, the train/test split results reveal that it still has its limitations. The model benefits from the additional features but remains affected by overfitting and the small sample size of only 32 teams. As before, more data or advanced modeling techniques may be needed in order to better understand and predict the complex nature of expected defensive points in the NFL.

## Ridge Regression: The Regularized Model

As the evaluation from the multiple regression model implied, the overall generalizability and prescence of multicollinearity in the model should be investigated further. Multicollinearity is an issue that arises in regression modeling where two or more independent variables are highly correlated, making it incredibly difficult to isolate each variable's unique effect on the dependent variable. Multicollinearity leads to unstable coefficients, inflated standard errors, and potentially interpretations of the model's performance.

Football statistics have some overlap in terms of how yards, points, etc. are connected. In order to combat the potential issue of multicollinearity in our model, we will integrate Ridge regression. 

Ridge applies L2 regularization, which penalizes large coefficients by adding the following term to the loss function:

$L(\beta) = \text{MSE} + \alpha \sum_{j=1}^p \beta_j^2$

The multiple regression model appeared highly effective overall without the need to remove features, so LASSO regularization was not used. Unlike Lasso (or L1 regularization), Ridge doesn't eliminate predictors entirely. Instead, the $ \alpha \sum_{j=1}^p \beta_j^2 $ term in the loss function shrinks all coefficients near, but not to zero. 

The hyperparameter $ \alpha $ controls the strength of the regularization penalty for error, with higher values meaning the penalty is more severe. For the purposes of our data, we will set $ \alpha $ = 1.0 in order to induce a moderate, but not overly significant penalty to the model.

Overall, Ridge lowers variance and improves prediction accuracy on new data, making it more robust, especially with many features. 

### Cross-Validation

Cross-Validation provides a more reliable measure for the model's performance out-of-sample than a single train-test split would, especially in a dataset with only 32 observations.

To evaluate the generalizability of the Ridge model, we conduct 5-fold cross-validation, measuring:

- Average $\text{MSE}$
- Standard deviation of $\text{MSE}$  
- Average \$R^2\$ 
- Standard deviation of \$R^2\$

These metrics help us assess both accuracy and stability. A higher average \$R^2\$ lends to better predictive performance, while a lower standard deviation implies a consistent performance across different data subsets. 

CV also enables comparison between Ridge and the ordinary multiple regression model. If Ridge produces improved \$R^2\$, lower $\text{MSE}$, or reduced variability, it would suggest that the regularization was successfully able to mitigate multicollinearity.

The following code trains the Ridge model with $ \alpha $ = 1.0, evaluates performance across five folds, and visualizes \$R^2\$ across folds.

In [ ]:
# Set our hyperparameter alpha  to 1
ridge = Ridge(alpha = 1.0)

# 5-fold cross-validation on the multivariate model
kf = KFold(n_splits = 5, shuffle = True, random_state = 42)

# Calculate the MSE and R^2 scores across folds
mse_scores = -cross_val_score(
    ridge, X_multi, nfl["EXP"], scoring = "neg_mean_squared_error", cv = kf
)
r2_scores = cross_val_score(
    ridge, X_multi, nfl["EXP"], scoring = "r2", cv = kf
)

# Display all the relevant statistics for our upcoming analysis
print("===== RIDGE REGRESSION (5-FOLD CV) =====")
print("Average MSE:", mse_scores.mean())
print("Std MSE:", mse_scores.std())
print("Average R²:", r2_scores.mean())
print("Std R²:", r2_scores.std())

# Display all the relevant model statistics for our upcoming analysis
folds = np.arange(1, 6)

plt.figure(figsize=(9,6))
plt.plot(folds, r2_scores, marker="o")
plt.axhline(np.mean(r2_scores), color = "red", linestyle="--", label = "Mean R²")

plt.title("Ridge Regression: R² Across 5 Folds")
plt.xlabel("Fold Number")
plt.ylabel("R² Score")
plt.legend()
plt.grid(alpha = 0.3)
plt.show()

**Figure 6.** Ridge regression \$R^2\$ scores across 5 cross-validation folds. The wide variation across folds, particularly the extreme negative outlier, reflects instability due to the small sample size and suggests that regularization didn't meaningfully improve predictive accuracy in this 
dataset.

## Interpretation of Ridge Regression Results

The average \$R^2\$ of approximately 0.049 suggest that the Ridge model explains very little of the variation in `EXP` on unseen data. While this is a slight improvement over earlier results that had a negative \$R^2\$, it still suggests that Ridge performs only marginally better than simply predicting the mean value of `EXP`. As a result, Ridge offers minimal improvement in predictive accuracy compared to the multiple model.

The standard deviation of \$R^2\$ is 0.636, which is considerably large. As shown in the visualization, Fold #5 had a strikingly low \$R^2\$ of almost -1.25. For reference, none of the other folds had an \$R^2\$ < -0.1. A value this extreme could be caused by the data in Fold #5 containing extreme outliers.

While this absurd fold value significantly influences the standard deviation, Ridge's performance fluctuates noticeably across all other folds as well. Some subsets have reasonable predictive strength, while others perform below expectations. This level of variability suggests that the model is still affected by the limited size of the `nfl` dataset and the irregular statistical patterns that are often seen in defensive stats.

The cross-validated $\text{MSE}$ of 1451.39 further confirms that the Ridge model doesn't meaningfully outperform the multiple regression model. The standard deviation value for $\text{MSE}$ of 528.072 reflects the same issues uncovered in the standard deviation for \$R^2\$: Fold #5 is impacting the score significantly, and the other folds still show some variation as well. While the L2 penalty stabilizes the coefficients and reduces the features' sensitivity to multicollinearity, it doesn't translate into any improvements in generalization for predicting `EXP`.

In context, these results reveal that regularization does not seem to significantly enhance model performance for this dataset. The multiple regression model, despite its mild overfitting, is still the strongest performer. Ridge regression succeeds in reducing the variance in coefficients but that didn't lend to any improvements in the accuracy of its predictions. As mentioned previously, this is likely due to the small sample size of NFL teams and the complex, noisy nature of defensive expected points.

## Discussion and Final Model Summary

This project examined how defensive takeaways influence Expected Points Contributed (`EXP`) across all 32 NFL teams in the 2024 season, using visualization, simple linear regression, multiple regression, and Ridge regression. Together, these analyses reveal a clear but nuanced picture of the true role takeaways play in influencing overall defensive performance. 

Below are my summarized findings for each model's performance, and how that speaks to the impact of defensive takeaways `TO` on Expected Points Contributed `EXP`:

### Simple Linear Regression

* Demonstrates the raw, unadjusted relationship between `TO` and `EXP`
* The positive slope of 5.69 indicates that each additional takeaway increases expected defensive points
* The in-sample \$R^2\$ = 0.328 suggests a moderate but meaningful relationship; explains roughly 33% of the variance in `EXP`

#### Train/Test Evaluation: Simple Linear Regression

* Revealed poor generalization performance for the simple model
* Out-of-sample \$R^2\$ = -0.631 shows that takeaways alone don't reliably predict `EXP` on new data
* This confirms that additional defensive metrics must be included to capture `EXP` in a more comprehensive manner

### Multiple Regression

* Controls for core defensive metrics such as points allowed `PA`, yards allowed `YDS`, first downs allowed `1stD`, and penalties `Pen`.
* The `TO` coefficient of 2.615 was by far the most significant explanatory feature, followed by the `1stD` coefficient of -1.168
* The model achieves a dramatically improved \$R^2\$ = 0.8213, explaining over 82% of the variance in `EXP`.
* The coefficient for `TO` is still positive and meaningful, which affirms that takeaways contribute independent value even after adjusting for more potential defensive factors

#### Train/Test Evaluation: Multiple Regression

* New features significantly improved the model's performance on unseen data (Test $\text{MSE}$ = 1289.94 vs 2979.75 for simple linear)
* Test \$R^2\$ = 0.294 explained less than 1/3 of variation of new data
* Since Train \$R^2\$ = 0.867 is vastly higher by comparison, the multiple model still overfit to the specific data
* A larger sample size would need to be utilized to gauge a better understanding if other factors would fill the gaps in unexplained variance

### Ridge Regression

* Used to stabilize the coefficient estimates in the presence of multicollinearity
* Cross-validated performance produced an average \$R^2\$ = 0.049, indicating an overall weak predictive improvement
* Fold #5 had an outlier \$R^2\$ of almost -1.25, which may have highly skewed the standard deviations of $\text{MSE}$= 528.072 and \$R^2\$ = 0.636
* Regularization didn't outperform the multiple regression model, likely due to the small sample size and correlated metrics reducing available signal

## Best Model

The multiple regression model was the best performing one! It explained approximately 82% of the variation in EXP, and was rich in features to create a solid, comprehensive predictive base given the data it had to work with. The simple linear model was largely ineffective, as it had too much bias as a result of using only one feature and being exposed to a small amount of data. The Ridge regression model turned out to be less useful in this modeling context than initially believed, as it had somewhat poorous predictive power and significant variation when conducting the 5-fold CV.

## Conclusion

The goal of this project was to answer the research question: How do takeaways (`TO`) impact the expected points contributed by all defense (`EXP`)?

Across all analyses conducted, the results consistently show that:

* Takeaways have a positive, meaningful effect on defensive expected points, as `EXP` was consistently shown to have a substantial increase with more `TO`
* The relationship is moderate when examined alone through simple linear regression
* When controlling for other defensive metrics, takeaways remain a significant independent predictor, and strongest of the statistics measured
* The best, highest performing model (multiple regression) explains over 82% of the variation in `EXP`
* The hypothesis that higher amounts of takeaways lead to higher expected defensive points is supported

While takeaways are clearly not the only factor influencing defensive performance, they play a major and quantifiable role. Teams that emphasize forcing turnovers can significantly improve their season-long defensive efficiency and contribution to scoring. From a football perspective, the statistical findings match the intuitive reasoning behind why takeaways are considered to be so important. As stated before, they halt offensive drives, often create favorable field position, eliminate scoring opportunities (at any part of the field), and can even directly create defensive touchdowns.

The analytical results from this study reaffirm the historical, long-standing belief in every level of football that defenses that force turnovers consistently give their team a better chance of preventing opposing points, thereby winning more games.

## Potential Improvements and Future Direction

Although the analysis provides plenty of valuable insight into how takeaways influence expected defensive points, several improvements and extensions could significantly strengthen the study. The 7 sections below discuss the limitations of the current study and different ideas to expand future work in analyzing NFL defensive statistics:

### 1. Expand the Dataset Across Multiple Seasons

This study only covered data from a single NFL season (2024). However, using only one NFL season limits the sample size, generalizability of models to new data, and the ability to evaluate trends over a series of time.  

Including 10–15 seasons of data would produce more statistical power via the rules of the Law of Large Numbers and/or the Central Limit Theorem. In addition, we could build more stable regression models that are less subject to bias (due to the increase in available training data), as well as a better understanding of whether the `TO` - `EXP` relationship is consistent over time.

### 2. Incorporate Advanced Defensive Metrics

Our analysis integrated the use of common football statistics like raw turnover numbers, penalties, 1st downs, etc. to compare to expected points contributed by defense `EXP`. To provide an even more in-depth analysis, we can use more modern football analytics include like Expected Points Added (EPA) per play, the success rate allowed, pressure rate, coverage grades, and turnover-worthy plays forced. These variables may explain defensive expected points more effectively than raw numbers would. For more insight on these statistics, visit the websites of Pro Football Reference and/or Pro Football Focus (PFF). The links are in the Resources section at the bottom of the page.

### 3. Feature Engineering

While our current study relies primarily on season-level aggregate statistics, any future work could benefit from developing additional derived variables that capture defensive performance with more nuance. Some examples include takeaways per game (which would standardize the statistic across seasons with differing amounts of games played), explosive plays allowed, Red Zone efficiency on defense, or even opponent-adjusted metrics that account for the strength of opposing offenses for each team's defense. Creating these higher-level, engineered features would provide more substantiative inputs for regression modeling and might be able to reveal potential relationships that aren't detectable through raw totals alone.

### 4. Explore Nonlinear or Interaction-Based Models

The linear models used in this project assume that defensive variables influence expected points in additive and proportional ways. However, NFL defenses are very complex systems in which certain variables may interact or behave nonlinearly. Future studies could utilize polynomial regression, interaction terms (for example, `TO × PA`), or even tree-based methods like random forests and gradient boosting. Using these more intricate, advanced modeling systems might uncover more nuanced patterns and better capture how different aspects of defensive performance combine to affect `EXP`.

### 5. Use Game-Level Rather Than Season-Level Data

Because this study used a single season's summary statistics for each team, a lot of the variation present in weekly performance isn't captured in the models. A more detailed analysis would use game-level data, which would greatly increase the sample size and enable even more advanced techniques such as mixed-effects modeling, time-series analysis, or drive-level expected points modeling. Additionally, game-level data accounts for differences in the length of seasons if using a wide range of data. Ever since 2021, NFL seasons have increased from 16 games to 17, and are subject to potentially increase to 18 in the near future. Using game-by-game data would mitigate this issue of differing season lengths and also provide deeper insight into when takeaways matter most and how their impact fluctuates over the course of an individual season.

### 6. Cross-Validation With Larger Samples

Although Ridge regression was implemented to reduce multicollinearity, the limited sample size (32 teams) constrained the effectiveness of the regularization process. With a larger dataset (see previous sections for insight into collecting more NFL data), CV metrics would become more stable, and regularization like Ridge or LASSO could truly be used more effectively, especially in predicting long-term trends. Larger samples would also make the comparisons between models more meaningful.

### 7. Incorporate Offensive Context

Defensive performance can't be fully separated from offensive factors. Time of possession, field position generated by the offense, turnover regression effects, etc. make up some of the underlying statistics that drive offensive performance, which has some effect on opposing defenses. Integrating offensive metrics into future analyses would help isolate the true defensive contribution to `EXP` and reduce omitted variable bias. Including potential model features like opponent offensive strength, average starting field position, or offensive scoring efficiency would produce even more accurate, layered estimates of how `TO` independently affect `EXP`.


### Summary

This project expresses plenty of meaningful insights into how takeaways affect the expected points contributed by all defense. To summarize, to make an even more insightful, data-driven analysis, I would look into expanding the dataset, refining features, and applying advanced modeling techniques. Using one or more of these approaches would significantly strengthen the connection between core football concepts and how measured NFL statistics reinforce their importance.

## References

Pro Football Reference. (2024). *2024 NFL Opponent Drive and Defense Statistics.*  
https://www.pro-football-reference.com/years/2024/opp.htm

## Resources
Pro Football Reference website (general page): https://www.pro-football-reference.com/

Pro Football Focus website: https://www.pff.com/